# Distributed Data Processing with PySpark — Problems and Solutions
**Technical Notes**  
*Rodrigo Kang*

This notebook contains complete solutions to the PySpark problems from the associated technical notes. The emphasis is not only on API syntax, but also on distributed-computing reasoning: lazy execution, partitions, shuffles, joins, skew, persistence, execution plans, Spark SQL, window functions, storage formats, and machine-learning workflows.

In [ ]:
# Core PySpark imports used throughout the notebook.
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

from pyspark import StorageLevel

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

In [ ]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .appName("PySparkProblems")
    .getOrCreate()
)

spark.version

## Problem 1 — Creating a Spark Session

### Problem

Create a local Spark session named `PySparkProblems` using all available local cores and inspect `spark.version`.

### Solution

`SparkSession` is the main entry point to modern Spark SQL/DataFrame functionality. It coordinates access to DataFrame creation, data sources, SQL execution, configuration, and the underlying Spark application context.

#### Implementation

In [ ]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .appName("PySparkProblems")
    .getOrCreate()
)

print(spark.version)

### Key Takeaway

The session is the user-facing gateway to Spark's structured-data engine; it is not itself the cluster.

## Problem 2 — Creating a DataFrame

### Problem

Create the supplied city-sales data as a Spark DataFrame, display it, and inspect the schema.

### Solution

`createDataFrame()` turns local Python records into a Spark DataFrame. The resulting object is a distributed logical data abstraction rather than a pandas-style in-memory table.

#### Implementation

In [ ]:
data = [
    (1, "Auckland", 120.0),
    (2, "Wellington", 95.0),
    (3, "Christchurch", 140.0),
    (4, "Auckland", 110.0)
]

columns = ["customer_id", "city", "sales"]

df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

### Key Takeaway

Spark DataFrames look tabular, but their operations build distributed execution plans rather than directly manipulating one local in-memory object.

## Problem 3 — Explicit Schema

### Problem

Recreate the DataFrame with an explicit schema and make `customer_id` non-nullable.

### Solution

An explicit schema documents the expected contract, avoids inference work, and reduces ambiguity about intended data types.

#### Implementation

In [ ]:
schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("city", T.StringType(), True),
    T.StructField("sales", T.DoubleType(), True),
])

df_schema = spark.createDataFrame(data, schema=schema)
df_schema.printSchema()

### Key Takeaway

Explicit schemas improve reliability and make the data contract visible.

## Problem 4 — Basic Inspection

### Problem

Inspect columns, schema, printed schema, rows, and row count. Identify which operations trigger computation.

### Solution

`columns`, `schema`, and `printSchema()` use structural metadata. `show()` and `count()` are actions that require data execution. On large data, actions can launch distributed jobs and should not be treated as free inspection.

#### Implementation

In [ ]:
print(df.columns)
print(df.schema)
df.printSchema()
df.show()
print(df.count())

### Key Takeaway

Surface simplicity does not imply low distributed cost; distinguish metadata access from actions.

## Problem 5 — Selecting Columns

### Problem

Select `customer_id` and `sales` with both string names and `F.col()`.

### Solution

`F.col("sales")` creates a Spark column expression. It does not materialise the entire column in the Python driver.

#### Implementation

In [ ]:
df.select("customer_id", "sales").show()

df.select(
    F.col("customer_id"),
    F.col("sales")
).show()

### Key Takeaway

Spark column objects are expression nodes used to build logical plans.

## Problem 6 — Column Expressions

### Problem

Create `sales_adjusted = 1.15 * sales` using `withColumn()`.

### Solution

The multiplication is expressed as a Spark column expression. Spark later evaluates it over distributed partitions; no explicit Python row loop is required.

#### Implementation

In [ ]:
df6 = df.withColumn(
    "sales_adjusted",
    F.col("sales") * F.lit(1.15)
)

df6.show()

### Key Takeaway

Prefer declarative column expressions to row-by-row Python logic.

## Problem 7 — Multiple Derived Columns

### Problem

Create `tax` and `sales_with_tax`.

### Solution

Each `withColumn()` returns a new logical DataFrame. The second expression can reference the column created by the previous transformation.

#### Implementation

In [ ]:
df7 = (
    df
    .withColumn(
        "tax",
        F.col("sales") * F.lit(0.15)
    )
    .withColumn(
        "sales_with_tax",
        F.col("sales") + F.col("tax")
    )
)

df7.show()

### Key Takeaway

Spark DataFrame transformations are immutable: each step describes a new dataset.

## Problem 8 — Filtering Rows

### Problem

Filter by sales and by sales plus city.

### Solution

Boolean Spark expressions are combined element-wise with `&`, `|`, and `~`. Parentheses are important because Python operator precedence can otherwise change the intended expression.

#### Implementation

In [ ]:
df.filter(
    F.col("sales") > 100
).show()

df.filter(
    (F.col("sales") > 100)
    & (F.col("city") == "Auckland")
).show()

### Key Takeaway

Filtering describes a distributed predicate; execution remains lazy until an action.

## Problem 9 — Conditional Columns

### Problem

Create `sales_category` with high, medium, and low values.

### Solution

`when()` constructs a Spark conditional expression evaluated inside the Spark engine.

#### Implementation

In [ ]:
df9 = df.withColumn(
    "sales_category",
    F.when(
        F.col("sales") >= 120,
        F.lit("high")
    )
    .when(
        F.col("sales") >= 100,
        F.lit("medium")
    )
    .otherwise(
        F.lit("low")
    )
)

df9.show()

### Key Takeaway

Conditional logic should remain in Spark expressions when possible.

## Problem 10 — String Operations

### Problem

Clean city names with whitespace and inconsistent case.

### Solution

Built-in string functions remain visible to Spark's optimiser and avoid the Python execution boundary introduced by a UDF.

#### Implementation

In [ ]:
dirty = spark.createDataFrame(
    [
        (1, "  auckland "),
        (2, "WELLINGTON"),
        (3, " christchurch")
    ],
    ["id", "city"]
)

clean = dirty.withColumn(
    "city_clean",
    F.initcap(
        F.lower(
            F.trim(F.col("city"))
        )
    )
)

clean.show()

### Key Takeaway

Use native Spark expressions before considering custom Python functions.

## Problem 11 — Missing Values

### Problem

Identify, remove, and fill missing values.

### Solution

Spark supplies distributed null-handling operations, but the choice to drop or impute values is still a statistical/business decision.

#### Implementation

In [ ]:
missing = spark.createDataFrame(
    [
        (1, "Auckland", 120.0),
        (2, None, 95.0),
        (3, "Wellington", None),
        (4, None, None)
    ],
    ["id", "city", "sales"]
)

missing.filter(
    F.col("sales").isNull()
).show()

missing.na.drop(
    subset=["sales"]
).show()

missing.na.fill({
    "city": "unknown"
}).show()

missing.na.fill({
    "sales": 0.0
}).show()

### Key Takeaway

Distributed execution changes scale, not the analytical meaning of missingness.

## Problem 12 — Duplicates

### Problem

Compare `distinct()` and `dropDuplicates()`, then deduplicate by customer ID.

### Solution

Both can remove duplicate rows. `dropDuplicates(subset)` lets the business key define duplication. Whether repeated keys are true duplicates depends on grain.

#### Implementation

In [ ]:
dups = spark.createDataFrame(
    [
        (1, "A", 10.0),
        (1, "A", 10.0),
        (1, "A", 12.0),
        (2, "B", 20.0)
    ],
    ["customer_id", "group", "amount"]
)

dups.distinct().show()
dups.dropDuplicates().show()
dups.dropDuplicates(["customer_id"]).show()

### Key Takeaway

Define duplicate semantics from the intended row grain, not appearance alone.

## Problem 13 — Sorting

### Problem

Sort by sales and city.

### Solution

A global sort can require repartitioning and ordering records across the cluster, making it potentially much more expensive than local sorting.

#### Implementation

In [ ]:
df.orderBy(
    F.col("sales").asc()
).show()

df.orderBy(
    F.col("sales").desc()
).show()

df.orderBy(
    F.col("city").asc(),
    F.col("sales").desc()
).show()

### Key Takeaway

Global ordering is a distributed coordination problem, not merely a local comparison operation.

## Problem 14 — Global Aggregation

### Problem

Calculate count, sum, mean, min, max, and standard deviation.

### Solution

Spark can compute local partial aggregates per partition and then combine them, reducing the amount of data that must be moved.

#### Implementation

In [ ]:
df.select(
    F.count("*").alias("n"),
    F.sum("sales").alias("sum_sales"),
    F.mean("sales").alias("mean_sales"),
    F.min("sales").alias("min_sales"),
    F.max("sales").alias("max_sales"),
    F.stddev("sales").alias("std_sales")
).show()

### Key Takeaway

Associative/combinable aggregates are naturally suited to distributed execution.

## Problem 15 — Grouped Aggregation

### Problem

Calculate grouped statistics by city.

### Solution

`groupBy()` defines groups; `agg()` supplies the group-level reductions. Spark may need to shuffle records so equal keys are brought together.

#### Implementation

In [ ]:
sales15 = spark.createDataFrame(
    [
        ("Auckland", 120.0),
        ("Auckland", 140.0),
        ("Wellington", 90.0),
        ("Wellington", 110.0),
        ("Christchurch", 80.0)
    ],
    ["city", "sales"]
)

(
    sales15
    .groupBy("city")
    .agg(
        F.count("*").alias("n"),
        F.sum("sales").alias("total_sales"),
        F.mean("sales").alias("mean_sales"),
        F.max("sales").alias("maximum_sale")
    )
    .show()
)

### Key Takeaway

Grouped aggregation changes grain and often requires key-based redistribution.

## Problem 16 — Grain after Aggregation

### Problem

See the problem statement in the associated technical notes.

### Solution

After grouping transaction-level data by `customer_id`, the result has one row per customer. The aggregation changes the observational unit from transactions to customers.

### Key Takeaway

Always restate the grain after an aggregation.

## Problem 17 — Transformations and Actions

### Problem

See the problem statement in the associated technical notes.

### Solution

`select`, `filter`, `withColumn`, `groupBy`, and `join` contribute to transformations/logical plans. `count`, `show`, `collect`, and `take` are actions that request concrete results. `groupBy()` returns a grouped object and becomes part of a transformation once followed by an aggregation.

### Key Takeaway

Lazy evaluation means transformations describe work; actions cause Spark to execute the required plan.

## Problem 18 — Lazy Evaluation

### Problem

See the problem statement in the associated technical notes.

### Solution

Before an action, Spark has generally built a logical plan rather than necessarily scanning all data. `result.count()` triggers planning and execution of the transformations required to produce the count.

### Key Takeaway

Spark separates describing a computation from executing it.

## Problem 19 — Reading an Execution Pipeline

### Problem

See the problem statement in the associated technical notes.

### Solution

`filter` and the arithmetic `withColumn` are typically narrow. `groupBy` is the likely shuffle boundary. The final grain is one row per `customer_id`.

### Key Takeaway

Find the grain and shuffle boundaries before tuning syntax.

## Problem 20 — Narrow versus Wide Transformations

### Problem

See the problem statement in the associated technical notes.

### Solution

`select`, `filter`, and many `withColumn` operations are typically narrow. `groupBy`, global `orderBy`, `distinct`, and `repartition` are wide. `join` can be wide but strategy-dependent, for example a broadcast join can avoid shuffling the large side.

### Key Takeaway

Wide transformations are important because they frequently introduce network redistribution.

## Problem 21 — Shuffle Reasoning

### Problem

See the problem statement in the associated technical notes.

### Solution

The two Auckland rows must contribute to the same city-level aggregate despite starting in separate partitions. Spark therefore redistributes key-related records or partial aggregates so the final Auckland result can be computed.

### Key Takeaway

A shuffle exists to satisfy dependencies that cannot be resolved entirely within original partitions.

## Problem 22 — Stages

### Problem

See the problem statement in the associated technical notes.

### Solution

The read, filter, and local column transformation can often be pipelined in one stage. The `groupBy`/shuffle creates a boundary; the post-shuffle aggregation occurs in a later stage.

### Key Takeaway

Shuffle boundaries are natural stage boundaries.

## Problem 23 — Basic Join

### Problem

Join customer-level and order-level DataFrames on `customer_id` and determine the resulting grain.

### Solution

The join enriches each order with customer attributes. Because one customer can have many orders but each order maps to one customer, the result remains one row per order.

#### Implementation

In [ ]:
customers23 = spark.createDataFrame(
    [
        (1, "Alpha"),
        (2, "Beta"),
        (3, "Gamma")
    ],
    ["customer_id", "customer_name"]
)

orders23 = spark.createDataFrame(
    [
        (101, 1, 120.0),
        (102, 2, 80.0),
        (103, 1, 95.0),
        (104, 3, 140.0)
    ],
    ["order_id", "customer_id", "amount"]
)

orders23.join(
    customers23,
    on="customer_id",
    how="left"
).select(
    "order_id",
    "customer_id",
    "customer_name",
    "amount"
).show()

### Key Takeaway

A many-to-one enrichment join preserves the grain of the many side.

## Problem 24 — Join Types

### Problem

Create unmatched keys and compare inner, left, right, and full joins.

### Solution

Inner keeps matches; left preserves all left rows; right preserves all right rows; full preserves the union of key populations.

#### Implementation

In [ ]:
customers24 = spark.createDataFrame(
    [(1, "Alpha"), (2, "Beta"), (3, "Gamma"), (4, "Delta")],
    ["customer_id", "customer_name"]
)

orders24 = spark.createDataFrame(
    [(101, 1, 120.0), (102, 2, 80.0), (103, 99, 50.0)],
    ["order_id", "customer_id", "amount"]
)

for how in ["inner", "left", "right", "full"]:
    print(how.upper())
    orders24.join(
        customers24,
        on="customer_id",
        how=how
    ).show()

### Key Takeaway

Join type is a population-retention decision.

## Problem 25 — Semi Join

### Problem

Return only customers with at least one matching order.

### Solution

`left_semi` performs an existence test while returning only the left-side columns, avoiding unnecessary materialisation of right-side attributes.

#### Implementation

In [ ]:
customers24.join(
    orders24,
    on="customer_id",
    how="left_semi"
).show()

### Key Takeaway

Use semi joins when the question is whether a match exists, not when right-side columns are needed.

## Problem 26 — Anti Join

### Problem

Return customers without matching orders.

### Solution

`left_anti` returns left-side rows for which no right-side key match exists.

#### Implementation

In [ ]:
customers24.join(
    orders24,
    on="customer_id",
    how="left_anti"
).show()

### Key Takeaway

Anti joins directly express absence-of-match business questions.

## Problem 27 — Join Cardinality

### Problem

See the associated technical-notes problem statement.

### Solution

`customers → orders` is one-to-many. Joining customer attributes onto the order table is many-to-one from the left-side order perspective and should preserve the order row count if customer keys are unique.

### Key Takeaway

Expected cardinality is a correctness invariant for joins.

## Problem 28 — Many-to-Many Row Multiplication

### Problem

See the associated technical-notes problem statement.

### Solution

Two matching left rows and three matching right rows produce six combinations. At Spark scale, accidental many-to-many joins can explode intermediate data, network traffic, and memory usage.

### Key Takeaway

Row multiplication is mathematically correct join behaviour and can still be operationally disastrous.

## Problem 29 — Join and Shuffle

### Problem

See the associated technical-notes problem statement.

### Solution

When both large sides are independently partitioned, records with equal `customer_id` may reside on different executors. A shuffle repartitions records by join key so matching rows can be colocated for the physical join.

### Key Takeaway

Large joins are frequently data-movement problems.

## Problem 30 — Broadcast Join

### Problem

See the associated technical-notes problem statement.

### Solution

Broadcasting the 200-row country table is a strong candidate because it is tiny relative to the transaction table. Each executor can receive the small table and join it locally against its large-side partitions.

### Key Takeaway

Broadcast the genuinely small side when executor memory and statistics make replication economical.

## Problem 31 — When Not to Broadcast

### Problem

See the associated technical-notes problem statement.

### Solution

If both sides are huge, replicating either dataset to every executor can exhaust executor memory and create substantial transfer cost. A distributed shuffle-based strategy is more appropriate.

### Key Takeaway

Broadcasting trades shuffle for replication; it is only beneficial when replication is cheap.

## Problem 32 — Logical Join versus Physical Strategy

### Problem

See the associated technical-notes problem statement.

### Solution

`left join` specifies which logical rows must be preserved. `broadcast hash join` describes a physical execution technique. Logical semantics and physical strategy are different layers.

### Key Takeaway

Separate what result is required from how Spark produces it.

## Problem 33 — Number of Partitions

### Problem

Inspect the current partition count and change it with `repartition()`.

### Solution

Partitions are Spark's principal units of parallel work. More partitions can expose more parallel tasks, but too many can increase overhead.

#### Implementation

In [ ]:
print("before:", df.rdd.getNumPartitions())

df33 = df.repartition(8)

print("after:", df33.rdd.getNumPartitions())

### Key Takeaway

Parallelism depends on partitions as well as available executor cores.

## Problem 34 — `repartition()` versus `coalesce()`

### Problem

See the associated technical-notes problem statement.

### Solution

`repartition(20)` redistributes data more fully and can rebalance partitions; `coalesce(20)` typically reduces partitions with a narrower dependency and less data movement. Coalesce may preserve imbalance, while repartition pays shuffle cost for better redistribution.

### Key Takeaway

Choose repartitioning based on the desired data distribution, not only the target count.

## Problem 35 — Repartitioning by Key

### Problem

See the associated technical-notes problem statement.

### Solution

`df.repartition(100, "customer_id")` hash-partitions by the key. This may reduce repeated future redistribution for compatible key-based operations, but the repartition itself is a shuffle and may create skew if the key distribution is uneven.

### Key Takeaway

A repartition should eliminate or reduce a justified downstream cost.

## Problem 36 — Too Few Partitions

### Problem

See the associated technical-notes problem statement.

### Solution

Eight partitions expose at most roughly eight concurrent partition tasks for that stage, leaving many of the 100 cores idle and making each partition extremely large.

### Key Takeaway

Too little partition parallelism underutilises the cluster.

## Problem 37 — Too Many Partitions

### Problem

See the associated technical-notes problem statement.

### Solution

One million tiny partitions create excessive scheduling, task-startup, metadata, and potentially output-file overhead relative to useful work.

### Key Takeaway

Parallelism has overhead; more partitions are not always better.

## Problem 38 — Data Skew

### Problem

See the associated technical-notes problem statement.

### Solution

A single extremely frequent customer key can concentrate a huge share of shuffle data in one partition or small set of partitions, producing long-running tasks and poor utilisation.

### Key Takeaway

Key-frequency imbalance translates into workload imbalance.

## Problem 39 — Straggler Task

### Problem

See the associated technical-notes problem statement.

### Solution

Likely causes include data skew, an oversized partition, slow/failed hardware or I/O, spill due to memory pressure, or unusually expensive records/UDF logic.

### Key Takeaway

A stage completes at the pace of its slowest required tasks.

## Problem 40 — More Executors

### Problem

See the associated technical-notes problem statement.

### Solution

Doubling executors does not halve runtime because only some work parallelises perfectly. Shuffle communication, skew, sequential stages, coordination, storage, and scheduling overhead remain.

### Key Takeaway

Distributed speedup is bounded by the non-parallel and communication-heavy parts of the workload.

## Problem 41 — Repeated Computation

### Problem

Explain why repeated actions can recompute an upstream lineage and how caching changes this.

### Solution

Without persistence, each action may evaluate the lineage required for its result. Caching materialises reusable partitions after the first action, so later actions can reuse the persisted result.

#### Implementation

In [ ]:
expensive41 = (
    df
    .filter(F.col("sales") > 0)
    .groupBy("city")
    .agg(F.sum("sales").alias("total_sales"))
)

# Without cache, these actions may cause repeated upstream evaluation.
expensive41.count()
expensive41.show()

### Key Takeaway

Lazy evaluation plus multiple actions can cause recomputation unless an intermediate is deliberately persisted.

## Problem 42 — Cache

### Problem

Cache an expensive DataFrame, execute actions, then unpersist it.

### Solution

Caching is justified when a costly result is reused enough that storing it is cheaper than recomputing its lineage.

#### Implementation

In [ ]:
cached42 = expensive41.cache()

cached42.count()   # materialises the cache
cached42.show()

cached42.unpersist()

### Key Takeaway

Caching is an optimisation for reuse, not a default step for every DataFrame.

## Problem 43 — When Not to Cache

### Problem

See the associated technical-notes problem statement.

### Solution

Use once → usually no; expensive join reused five times → likely yes; trivial transformation → usually no; enormous DataFrame consuming nearly all memory → likely no unless storage and reuse economics clearly justify it.

### Key Takeaway

Cache based on reuse and recomputation cost under real resource constraints.

## Problem 44 — `collect()`

### Problem

See the associated technical-notes problem statement.

### Solution

`collect()` materialises every result row on the driver as local Python objects. A cluster may process a dataset whose total size greatly exceeds driver memory, so collecting all rows can fail even after distributed computation succeeds.

### Key Takeaway

Distributed processability does not imply driver collectability.

## Problem 45 — `toPandas()`

### Problem

See the associated technical-notes problem statement.

### Solution

Two thousand rows by twenty columns is usually a reasonable candidate. Two hundred million rows is not, because `toPandas()` moves the result into one driver process and therefore returns to single-machine memory limits.

### Key Takeaway

Convert to pandas only after Spark has reduced data to a safely local size.

## Problem 47 — Filter Early

### Problem

See the associated technical-notes problem statement.

### Solution

Filter the transaction side before the join when the predicate is logically independent of the product table. This reduces rows entering the expensive join and can reduce shuffle, I/O, and memory costs.

### Key Takeaway

Reduce data volume before expensive distributed operations when semantics allow it.

## Problem 48 — Column Pruning

### Problem

See the associated technical-notes problem statement.

### Solution

Select only `customer_id`, `product_id`, and `amount`. Carrying less data can reduce scan, serialisation, memory, and shuffle volume.

### Key Takeaway

Reduce width as well as row count when unnecessary columns would otherwise flow through expensive stages.

## Problem 49 — Built-In Function versus UDF

### Problem

See the associated technical-notes problem statement.

### Solution

Prefer `F.upper(F.col("city"))`. Spark can analyse and optimise the built-in expression, while a Python UDF crosses into opaque Python execution and adds serialisation overhead.

### Key Takeaway

Native expressions preserve optimiser visibility.

## Problem 50 — Appropriate Use of a Python UDF

### Problem

See the associated technical-notes problem statement.

### Solution

A UDF may be justified for genuinely domain-specific Python logic unavailable through Spark SQL functions, such as applying a proprietary parsing algorithm or specialised Python package. Even then, first check whether native or vectorised alternatives exist.

### Key Takeaway

UDF support is an escape hatch, not the default transformation mechanism.

## Problem 46 — Reading an Execution Plan

### Problem

Create a grouped aggregation and inspect `explain(mode="formatted")`.

### Solution

Look for scan, filter/project, aggregate, and `Exchange` nodes. An `Exchange` generally signals repartitioning/data redistribution, commonly a shuffle.

#### Implementation

In [ ]:
plan46 = (
    df
    .filter(F.col("sales") > 0)
    .groupBy("city")
    .agg(F.sum("sales").alias("total_sales"))
)

plan46.explain(mode="formatted")

### Key Takeaway

Execution plans connect declarative code to physical distributed operators.

## Problem 51 — Spark SQL

### Problem

Create a temporary view and compute total sales by city using SQL and DataFrame API.

### Solution

Both interfaces ultimately use Spark's structured query engine. The choice is mainly about expressiveness and maintainability, not a fundamentally separate execution backend.

#### Implementation

In [ ]:
df.createOrReplaceTempView("sales")

spark.sql(
    '''
    SELECT
        city,
        SUM(sales) AS total_sales
    FROM sales
    GROUP BY city
    '''
).show()

(
    df
    .groupBy("city")
    .agg(F.sum("sales").alias("total_sales"))
    .show()
)

### Key Takeaway

Spark SQL and the DataFrame API are two declarative surfaces over the same structured engine.

## Problem 52 — SQL Join

### Problem

Join `customers` and `orders` temporary views with SQL.

### Solution

Spark SQL returns another Spark DataFrame, so subsequent processing can continue through either SQL or the DataFrame API.

#### Implementation

In [ ]:
customers23.createOrReplaceTempView("customers")
orders23.createOrReplaceTempView("orders")

sql_join52 = spark.sql(
    '''
    SELECT
        o.order_id,
        c.customer_name,
        o.amount
    FROM orders AS o
    LEFT JOIN customers AS c
        ON o.customer_id = c.customer_id
    '''
)

sql_join52.show()

### Key Takeaway

SQL results remain distributed Spark DataFrames.

## Problem 53 — Window versus Aggregation

### Problem

Compare a customer total created with a window against one created with `groupBy()`.

### Solution

The window retains transaction grain and repeats the customer total across each customer's rows. `groupBy()` reduces the dataset to customer grain.

#### Implementation

In [ ]:
transactions53 = spark.createDataFrame(
    [
        (1, 101, 10.0),
        (1, 102, 20.0),
        (2, 201, 30.0)
    ],
    ["customer_id", "transaction_id", "amount"]
)

w53 = Window.partitionBy("customer_id")

transactions53.withColumn(
    "customer_total",
    F.sum("amount").over(w53)
).show()

transactions53.groupBy(
    "customer_id"
).agg(
    F.sum("amount").alias("customer_total")
).show()

### Key Takeaway

Windows compute group-relative values without reducing row count.

## Problem 54 — Row Number

### Problem

Number each customer's orders chronologically.

### Solution

Partition by customer and order by date; `row_number()` restarts for each customer.

#### Implementation

In [ ]:
orders54 = spark.createDataFrame(
    [
        (1, "2026-01-03"),
        (1, "2026-01-01"),
        (2, "2026-02-01")
    ],
    ["customer_id", "order_date"]
).withColumn(
    "order_date",
    F.to_date("order_date")
)

w54 = Window.partitionBy(
    "customer_id"
).orderBy(
    "order_date"
)

orders54.withColumn(
    "order_number",
    F.row_number().over(w54)
).show()

### Key Takeaway

Ordered windows express sequence within groups while preserving row grain.

## Problem 55 — Ranking

### Problem

Rank transactions by sales within city and compare `rank()` with `dense_rank()`.

### Solution

`rank()` leaves gaps after ties; `dense_rank()` does not.

#### Implementation

In [ ]:
rank55 = spark.createDataFrame(
    [
        ("Auckland", 100.0),
        ("Auckland", 100.0),
        ("Auckland", 90.0),
        ("Wellington", 80.0)
    ],
    ["city", "sales"]
)

w55 = Window.partitionBy(
    "city"
).orderBy(
    F.col("sales").desc()
)

rank55.select(
    "*",
    F.rank().over(w55).alias("rank"),
    F.dense_rank().over(w55).alias("dense_rank")
).show()

### Key Takeaway

Ranking semantics matter when ties occur.

## Problem 56 — Lag

### Problem

Create previous amount and amount change by customer and date.

### Solution

`lag()` returns null for the first row of each ordered partition because no preceding observation exists.

#### Implementation

In [ ]:
tx56 = spark.createDataFrame(
    [
        (1, "2026-01-01", 10.0),
        (1, "2026-01-03", 15.0),
        (2, "2026-01-02", 8.0)
    ],
    ["customer_id", "date", "amount"]
).withColumn(
    "date",
    F.to_date("date")
)

w56 = Window.partitionBy(
    "customer_id"
).orderBy(
    "date"
)

(
    tx56
    .withColumn(
        "previous_amount",
        F.lag("amount", 1).over(w56)
    )
    .withColumn(
        "amount_change",
        F.col("amount") - F.col("previous_amount")
    )
    .show()
)

### Key Takeaway

Lag features encode within-entity temporal differences.

## Problem 57 — Running Total

### Problem

Create cumulative sales per customer.

### Solution

The window includes every prior row up to the current row, so the result is the prefix sum within each customer.

#### Implementation

In [ ]:
w57 = (
    Window
    .partitionBy("customer_id")
    .orderBy("date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

tx56.withColumn(
    "cumulative_sales",
    F.sum("amount").over(w57)
).show()

### Key Takeaway

Running aggregates preserve row grain while accumulating ordered history.

## Problem 58 — Window Performance

### Problem

See the associated technical-notes problem statement.

### Solution

Partitioned ordered windows may require a shuffle to bring rows for each key together and then a sort within partitions, so they can be expensive on large datasets.

### Key Takeaway

Window semantics often imply both distribution and ordering costs.

## Problem 60 — CSV versus Parquet

### Problem

See the associated technical-notes problem statement.

### Solution

For repeated large-scale analytics, Parquet is generally preferable because it preserves schema, is columnar, supports compression and column pruning, and integrates well with predicate pushdown. CSV remains useful for simple interchange and human readability.

### Key Takeaway

Choose storage formats according to analytical access patterns, not familiarity alone.

## Problem 62 — Bad Partition Column

### Problem

See the associated technical-notes problem statement.

### Solution

A nearly unique `transaction_id` is usually a poor physical partition key because it can create extremely many tiny directories/files with little pruning benefit.

### Key Takeaway

Good physical partition columns have useful filtering semantics and moderate cardinality.

## Problem 63 — Small Files Problem

### Problem

See the associated technical-notes problem statement.

### Solution

Millions of tiny files create planning, metadata, file-open, and task-scheduling overhead even when total bytes are manageable.

### Key Takeaway

File count is a systems characteristic, not merely a cosmetic storage detail.

## Problem 64 — Arrow

### Problem

See the associated technical-notes problem statement.

### Solution

Arrow can improve columnar data transfer between Spark and Python/pandas, but it does not change the fact that `toPandas()` materialises the entire result in driver memory.

### Key Takeaway

Faster transfer is not the same as unlimited local capacity.

## Problem 59 — Writing Parquet

### Problem

Write a DataFrame in overwrite mode and explain the output structure.

### Solution

Spark writes distributed output as a directory containing part files, commonly one or more per output partition.

#### Implementation

In [ ]:
(
    df
    .write
    .mode("overwrite")
    .parquet("output/sales")
)

### Key Takeaway

Multiple output files are a natural consequence of distributed partitioned writing.

## Problem 61 — Partitioned Output

### Problem

Write transactions partitioned by year and month.

### Solution

Physical partitioning can allow later time-filtered reads to skip unrelated directories through partition pruning.

#### Implementation

In [ ]:
transactions61 = spark.createDataFrame(
    [
        (1, 2026, 1, 100.0),
        (2, 2026, 2, 120.0),
        (3, 2025, 12, 90.0)
    ],
    ["id", "year", "month", "amount"]
)

(
    transactions61
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet("output/transactions")
)

### Key Takeaway

Physical partitioning should support common selective access patterns.

## Problem 65 — Pandas UDF

### Problem

Create a Pandas UDF that returns `x - mean(x)`.

### Solution

A Pandas UDF receives vectorised pandas batches and uses Arrow-based exchange rather than invoking Python once per Spark row.

#### Implementation

In [ ]:
import pandas as pd

@F.pandas_udf("double")
def centre_values(values: pd.Series) -> pd.Series:
    return values - values.mean()

### Key Takeaway

Vectorised UDFs reduce some Python boundary overhead, but native Spark functions remain preferable when available.

## Problem 66 — PySpark or pandas?

### Problem

Choose the more natural tool for several workloads.

### Solution

200 MB interactive laptop analysis → pandas; 15 TB distributed dataset → PySpark; 10,000-row final aggregate → pandas is often natural; multi-terabyte joins → PySpark; publication-quality plot → usually pandas/matplotlib after reduction; summaries from an already-local pandas DataFrame → pandas.

### Key Takeaway

Choose the simplest execution model that comfortably handles the data and workflow.

## Problem 67 — Building a Modelling Table

### Problem

Describe a Spark pipeline producing one row per customer from customers, orders, and order lines.

### Solution

Aggregate order lines to order-level totals first if needed, preserving one row per order after merging with `orders`. Then group by `customer_id` to compute order count, total units, and spending. Finally left-join the customer-level features into `customers`. Track grain explicitly at each step.

### Key Takeaway

Feature engineering is safest when every intermediate table has an explicit grain.

## Problem 68 — Leakage in Distributed Feature Engineering

### Problem

Explain why future spending cannot be used for a January 1 prediction.

### Solution

The feature contains information not available at prediction time. Spark can compute it perfectly and still produce a statistically invalid feature; distributed correctness does not enforce temporal causality.

### Key Takeaway

Computational correctness and statistical validity are separate requirements.

## Problem 69 — `VectorAssembler`

### Problem

Combine age, income, and tenure into a `features` vector.

### Solution

Spark ML algorithms commonly consume one vector-valued features column rather than many independent scalar predictor columns.

#### Implementation

In [ ]:
ml69 = spark.createDataFrame(
    [
        (25.0, 42000.0, 2.0, 0.0),
        (31.0, 51000.0, 5.0, 1.0),
        (28.0, 47000.0, 3.0, 0.0)
    ],
    ["age", "income", "tenure", "target"]
)

assembler69 = VectorAssembler(
    inputCols=["age", "income", "tenure"],
    outputCol="features"
)

assembler69.transform(ml69).show(truncate=False)

### Key Takeaway

VectorAssembler maps tabular predictor columns into Spark ML's expected feature-vector representation.

## Problem 70 — Estimator and Model

### Problem

Identify the estimator/model lifecycle for linear regression.

### Solution

`LinearRegression(...)` is an Estimator containing configuration. `fit(training_data)` learns parameters and returns a fitted `LinearRegressionModel`, whose learned state is used later by `transform()` to add predictions.

#### Implementation

In [ ]:
regression70 = LinearRegression(
    featuresCol="features",
    labelCol="target"
)

training70 = assembler69.transform(ml69)

model70 = regression70.fit(training70)
model70.transform(training70).show(truncate=False)

### Key Takeaway

An Estimator learns state; the fitted Model is a Transformer.

## Problem 71 — Spark ML Pipeline

### Problem

Create a pipeline containing `VectorAssembler` and `LinearRegression`, fit it, and transform test data.

### Solution

During `Pipeline.fit()`, transformer stages are applied and estimator stages are fitted. The result is a `PipelineModel` containing fitted stages.

#### Implementation

In [ ]:
assembler71 = VectorAssembler(
    inputCols=["age", "income", "tenure"],
    outputCol="features"
)

regression71 = LinearRegression(
    featuresCol="features",
    labelCol="target"
)

pipeline71 = Pipeline(
    stages=[
        assembler71,
        regression71
    ]
)

pipeline_model71 = pipeline71.fit(ml69)
pipeline_model71.transform(ml69).show(truncate=False)

### Key Takeaway

Spark ML pipelines package repeatable preprocessing and fitted modelling state.

## Problem 72 — Distributed Data Preparation versus Distributed Training

### Problem

See the associated technical-notes problem statement.

### Solution

No. Spark can reduce 10 TB to a 500,000-row modelling table, which may then be exported to a local or specialised framework such as scikit-learn or XGBoost if that better fits the algorithm and environment.

### Key Takeaway

Distributed preprocessing does not imply distributed model training.

## Problem 73 — Diagnose a Poor Spark Workflow

### Problem

See the associated technical-notes problem statement.

### Solution

`collect()` pulls all distributed rows into the driver and the Python loop then performs local processing, defeating Spark's distributed execution. Keep transformations expressed through DataFrame/SQL operations or suitable distributed UDF mechanisms.

### Key Takeaway

Do not collapse a distributed workload to the driver before the heavy computation is complete.

## Problem 74 — Diagnose Repeated Actions

### Problem

See the associated technical-notes problem statement.

### Solution

Each action can trigger a separate Spark job and may re-evaluate shared lineage. Caching may help only if the reused base DataFrame is expensive to recompute and reused enough; inspect lineage, source cost, memory availability, and action plans first.

### Key Takeaway

Count actions and evaluate reuse before persisting.

## Problem 75 — Diagnose Unnecessary Repartitioning

### Problem

See the associated technical-notes problem statement.

### Solution

Each `repartition()` can introduce a shuffle. Repartitioning three times without a distribution requirement may move the same data repeatedly for no benefit.

### Key Takeaway

Partition changes should serve a clear downstream need.

## Problem 76 — Diagnose a Join Strategy

### Problem

See the associated technical-notes problem statement.

### Solution

Investigate a broadcast join for the 20 KB country-code table. Verify its actual size, executor memory, statistics, and whether the plan/auto-broadcast settings already support the strategy.

### Key Takeaway

When one side is tiny, ask why both sides are being shuffled.

## Problem 77 — Diagnose Skew

### Problem

See the associated technical-notes problem statement.

### Solution

The 30 GB tasks indicate extreme partition imbalance, likely caused by highly frequent or otherwise skewed join-key values.

### Key Takeaway

Inspect key-frequency distributions when a small subset of tasks dominate shuffle stages.

## Problem 78 — Execution Plan Reasoning

### Problem

See the associated technical-notes problem statement.

### Solution

`Scan` reads source data; `Filter` applies a predicate; `Project` selects/derives columns; `Exchange` redistributes data; `HashAggregate` performs aggregation. `Exchange` is the clearest shuffle signal.

### Key Takeaway

Learn to read the physical plan structurally before memorising every operator.

## Problem 79 — End-to-End Spark Reasoning

### Problem

Explain the supplied select/filter/broadcast-join/groupBy/cache pipeline.

### Solution

The initial grain is one row per transaction. Selecting only required columns reduces width. The positive-amount filter is typically narrow. Broadcasting `products` is sensible only if it is genuinely small. A many-to-one product lookup join should preserve transaction grain if `product_id` is unique in `products`; duplicate product keys could multiply rows. `groupBy(customer_id)` introduces key-based redistribution and changes grain to one row per customer. `countDistinct(product_id)` requires more coordination than an ordinary row count because duplicates must be resolved across partitions. Caching is justified only if the customer summary will be reused. The cache is populated by the first action such as `count()`, `show()`, or a write.

### Key Takeaway

Explain Spark pipelines simultaneously in terms of grain, logical transformations, and physical data movement.

## Problem 80 — Complete Architecture Exercise

### Problem

Design a monthly customer-level churn feature pipeline from multi-terabyte Parquet sources.

### Solution

Read only the required transaction, customer, and product columns from Parquet. Apply valid date, amount, and population filters as early as possible. Track grains: transaction, customer, and product tables initially; transaction-product after lookup joins; customer-month after aggregation. Validate expected many-to-one dimension joins conceptually and broadcast only genuinely small lookup tables. Expect shuffles around large joins, groupBy, distinct counts, and some windows. Investigate skew in customer or product keys. Aggregate transaction count, total/average spending, and distinct products by customer and month, then enrich with customer age/region. Cache only if the expensive customer-month table feeds several downstream actions. Write Parquet, physically partitioned by a coarse time key such as year/month if it matches query patterns and avoids excessive small files. Move to pandas only if the final modelling table is safely local. Spark ML is optional; a local/specialised framework may be better after reduction. Prevent temporal leakage by constructing every monthly feature from data available up to the prediction cutoff. If performance is poor, inspect scans, pushed filters, exchanges, join strategies, partition sizes, skew, spill, and repeated lineage in `explain()` and runtime metrics.

### Key Takeaway

A scalable architecture is one that is both statistically valid and explicit about where data moves, aggregates, and changes grain.

# Final Review

The PySpark problems in this notebook intentionally combine API usage with distributed-systems reasoning.

The most important ideas are:

- Spark DataFrames are distributed logical data abstractions, not simply pandas DataFrames with larger capacity.
- Transformations build plans lazily; actions trigger execution.
- Partitions determine units of parallel work.
- Narrow transformations can often proceed locally within partitions, while wide transformations may require shuffles.
- Shuffles, large scans, and data movement frequently dominate arithmetic cost.
- Join correctness depends on grain and cardinality; join performance depends additionally on physical strategy.
- Broadcast joins can avoid large-side shuffles when one side is genuinely small.
- Repartitioning, caching, and UDFs are tools to use deliberately, not automatic optimisation steps.
- Skew can make a small number of tasks dominate an entire stage.
- `collect()` and `toPandas()` move distributed results back into one driver process and therefore restore single-machine memory constraints.
- Spark SQL, the DataFrame API, window functions, and Spark ML all operate within the same structured distributed-computation model.
- Parquet and sensible output partitioning are usually more appropriate than forcing distributed analytical output into local single-file conventions.
- Spark is often most valuable in machine learning for large-scale data preparation, joins, aggregation, and feature construction, even when final model training occurs elsewhere.
- Distributed computation does not prevent target leakage, invalid grains, or poor feature definitions.

A strong PySpark practitioner therefore asks two questions at every stage:

1. **Does this transformation produce the analytically correct dataset?**
2. **What data movement and execution structure does Spark require to produce it?**